In [0]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
titles_emotions_path = PROJECT_ROOT / "data" / "processed" / "05_titles_emotion_scores.csv"

df = pd.read_csv(titles_emotions_path)

# select region, count(*) where dominant_emotion is not null
region_counts = df[df['dominant_emotion'].notnull()]['region'].value_counts()

print("Region Counts:")
print(region_counts)

display(df.head())

In [0]:
# top regions of each emotion_score
for emotion in df.columns[6:]:
    print(f"Top regions for {emotion}:")
    top_regions = df.groupby('region')[emotion].mean().sort_values(ascending=False)
    print(top_regions)
    print("\n")

In [0]:
#top emotions for each region
for region in df['region'].unique():
    print(f"Top emotions for {region}:")
    region_emotions = df[df['region'] == region].iloc[:, 6:].mean().sort_values(ascending=False)
    print(region_emotions)
    print("\n")

In [0]:
import plotly.express as px
import plotly.graph_objects as go

emotion_cols = df.columns[6:]
heatmap_data = df.groupby('region')[emotion_cols].mean()

fig_heatmap = px.imshow(
    heatmap_data.T,
    labels=dict(x="Region", y="Emotion", color="Mean Score"),
    x=heatmap_data.index,
    y=emotion_cols,
    color_continuous_scale="RdYlGn",
    aspect="auto",
    title="Emotion Intensity by Region (Heatmap)"
)
fig_heatmap.update_layout(height=500, width=1000)
fig_heatmap.show()

In [0]:
# Bubble Chart: Compare two emotions across regions
emotion_cols_list = list(emotion_cols)

# Use first two emotions for comparison
emotion1, emotion2 = emotion_cols_list[0], emotion_cols_list[1]

# Group by region and calculate mean scores + count of rows
bubble_means = df.groupby('region')[[emotion1, emotion2]].mean().reset_index()
bubble_counts = df.groupby('region').size().reset_index(name='song_count')
bubble_data = bubble_means.merge(bubble_counts, on='region', how='left')

fig_bubble = px.scatter(
    bubble_data,
    x=emotion1,
    y=emotion2,
    size='song_count',
    hover_data={'region': True, 'song_count': True},
    color='region',
    size_max=50,
    title=f"Bubble Chart: {emotion1} vs {emotion2} by Region",
    labels={emotion1: emotion1.replace('_', ' ').title(),
            emotion2: emotion2.replace('_', ' ').title()},
)
fig_bubble.update_layout(height=600, width=900)
fig_bubble.show()

In [0]:
# Bar Charts: Top regions for each emotion
emotion_cols_list = list(emotion_cols)
num_emotions = len(emotion_cols_list)

# Create subplots
from plotly.subplots import make_subplots

fig_bars = make_subplots(
    rows=(num_emotions + 2) // 3,
    cols=3,
    subplot_titles=emotion_cols_list,
    specs=[[{"type": "bar"} for _ in range(3)] for _ in range((num_emotions + 2) // 3)]
)

for idx, emotion in enumerate(emotion_cols_list):
    row = (idx // 3) + 1
    col = (idx % 3) + 1
    
    top_regions = df.groupby('region')[emotion].mean().sort_values(ascending=False).head(5)
    
    fig_bars.add_trace(
        go.Bar(x=top_regions.index, y=top_regions.values, name=emotion, showlegend=False),
        row=row, col=col
    )
    fig_bars.update_yaxes(title_text=emotion.replace('_', ' ').title(), row=row, col=col)

fig_bars.update_layout(height=300 * ((num_emotions + 2) // 3), width=1200, 
                       title_text="Top 5 Regions for Each Emotion (Bar Charts)")
fig_bars.show()

In [0]:
# Box Plots: Distribution of emotion scores by region
# Prepare data in long format for box plots
df_melted = df[['region'] + list(emotion_cols)].melt(id_vars='region', 
                                                       var_name='emotion', 
                                                       value_name='score')

fig_box = px.box(
    df_melted,
    x='emotion',
    y='score',
    color='emotion',
    facet_col='region',
    facet_col_wrap=3,
    title="Distribution of Emotion Scores by Region (Box Plots)",
    labels={'score': 'Score', 'emotion': 'Emotion'},
    height=800,
    width=1200
)
fig_box.show()

In [0]:
# Sunburst Chart: Hierarchical view (Region -> Emotion)
# Calculate mean scores by region and emotion
sunburst_data = df[['region'] + list(emotion_cols)].melt(
    id_vars='region',
    var_name='emotion',
    value_name='score'
)
sunburst_agg = sunburst_data.groupby(['region', 'emotion'], as_index=False)['score'].mean()

fig_sunburst = px.sunburst(
    sunburst_agg,
    path=['region', 'emotion'],
    values='score',
    color='score',
    color_continuous_scale='RdYlGn',
    title='Sunburst Chart: Emotions by Region',
    height=700,
    width=900
)
fig_sunburst.show()

In [0]:
# ── Differential Heatmap: what makes each region distinctive ─────────────────
# Shows each region's mean emotion score MINUS the global mean.
# Positive = above global average; negative = below.
# This exposes regional character even when scores are generally high.

global_mean = df[emotion_cols].mean()
region_means = df.groupby('region')[emotion_cols].mean()
diff = region_means.subtract(global_mean)

# Rename columns for display
diff.columns = [c.replace('emotion_', '') for c in diff.columns]

fig_diff = px.imshow(
    diff.T,
    labels=dict(x="Region", y="Emotion", color="vs Global Mean"),
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Regional Emotion Character — Deviation from Global Mean<br>"
          "<sup>Positive (red) = above average for that region; Negative (blue) = below</sup>",
)
fig_diff.update_layout(height=500, width=1000)
fig_diff.show()